In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q07/annotated/checkpoints/pre_cell_5.pickle

trying: ['load_customer']
me:  4
trying: ['nation']


me:  5
trying: ['STORAGE_OPTS']
me:  0
trying: ['tpch_parent']
me:  0
trying: ['DATA_ROOT']
me:  0
trying: ['supplier']
me:  2
trying: ['customer']
me:  4
trying: ['pd']
me:  0
trying: ['load_nation']
me:  5
trying: ['load_orders']
me:  3
trying: ['lineitem']


me:  1
trying: ['load_supplier']
me:  2
trying: ['load_lineitem']
me:  1
trying: ['orders']
me:  3


Declaring variable STORAGE_OPTS
Declaring variable tpch_parent
Declaring variable DATA_ROOT
Declaring variable pd
Declaring variable lineitem
Declaring variable load_lineitem
Declaring variable supplier
Declaring variable load_supplier
Declaring variable load_orders
Declaring variable orders
Declaring variable load_customer
Declaring variable customer
Declaring variable nation
Declaring variable load_nation


In [4]:
%%RecordEvent
%%time
### cell 5 ###

# 1. Filter & project lineitem on the GPU using string dates (avoids pd.Timestamp)
mask = (lineitem.L_SHIPDATE >= "1995-01-01") & (lineitem.L_SHIPDATE < "1997-01-01")
li = lineitem[mask][[
    "L_ORDERKEY", "L_SUPPKEY", "L_SHIPDATE", "L_EXTENDEDPRICE", "L_DISCOUNT"
]]
# extract year using dt.year
li["L_YEAR"] = li.L_SHIPDATE.dt.year.astype("int16")
# compute volume
li["VOLUME"] = li.L_EXTENDEDPRICE * (1 - li.L_DISCOUNT)
# keep only the needed columns
li = li[["L_ORDERKEY", "L_SUPPKEY", "L_YEAR", "VOLUME"]]

# 2. Pre-filter nation table once for both FRANCE and GERMANY
nations = nation[nation.N_NAME.isin(["FRANCE", "GERMANY"])][["N_NATIONKEY", "N_NAME"]]

# 3. Attach nation names to customer and supplier
cust = (
    customer
    .merge(nations, left_on="C_NATIONKEY", right_on="N_NATIONKEY", how="inner")[["C_CUSTKEY", "N_NAME"]]
    .rename(columns={"N_NAME": "CUST_NATION"})
)
supp = (
    supplier
    .merge(nations, left_on="S_NATIONKEY", right_on="N_NATIONKEY", how="inner")[["S_SUPPKEY", "N_NAME"]]
    .rename(columns={"N_NAME": "SUPP_NATION"})
)

# 4. Join lineitem → orders → customer → supplier, then filter the two nation pairs
df = (
    li
    .merge(orders[["O_ORDERKEY", "O_CUSTKEY"]], left_on="L_ORDERKEY", right_on="O_ORDERKEY", how="inner")
    .merge(cust,   left_on="O_CUSTKEY", right_on="C_CUSTKEY", how="inner")
    .merge(supp,   left_on="L_SUPPKEY", right_on="S_SUPPKEY", how="inner")[["SUPP_NATION", "CUST_NATION", "L_YEAR", "VOLUME"]]
)

df = df[
    ((df.SUPP_NATION == "FRANCE") & (df.CUST_NATION == "GERMANY"))
    | ((df.SUPP_NATION == "GERMANY") & (df.CUST_NATION == "FRANCE"))
]

# 5. Aggregate on GPU and rename
total = (
    df
    .groupby(["SUPP_NATION", "CUST_NATION", "L_YEAR"], as_index=False)["VOLUME"]
    .sum()
    .rename(columns={"VOLUME": "REVENUE"})
)

CPU times: user 92 ms, sys: 56 ms, total: 148 ms
Wall time: 152 ms


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q07/rewritten/o4_mini_high/checkpoints/post_cell_5_try_2.pickle

migration speed (bps): 809040625.7216026
---------------------------
variables to migrate:
load_lineitem 144
nations 48
load_orders 144
df 48
orders 48
tpch_parent 54
supp 48
total 48
customer 48
load_customer 144
cust 48
mask 48
nation 48
load_nation 144
li 48
pd 72
load_supplier 144
STORAGE_OPTS 64
supplier 48
DATA_ROOT 80
lineitem 48
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/q07/rewritten/o4_mini_high/checkpoints/post_cell_5_try_2.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['lineitem']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables ['supplier']
Future variables ['lineitem']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables ['orders']
Future variables ['lineitem', 'supplier']
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables ['customer']
Intermediate variables []
Future variables ['lineitem', 'orders', 'supplier']
Modified dataframes
======= Cell 4 =======
Input variables []
Active variables []
Intermediate variables ['nation']
Future variables ['lineitem', 'orders', 'customer', 'supplier']
Modified dataframes
======= Cell 5 =======
Input variables []
Active variables []
Intermediate variables ['nations', 'mask', 'df', 'cust', 'li', 'total', 'supp']
Future variables []
Modified dataframes
Saved cell execution inf

In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/q07/opt_cell_exec_info_5_try_2.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[5], f)


In [8]:
opt_output = Out.get(4)

In [9]:
total_opt = total
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q07/annotated/checkpoints/post_cell_5.pickle

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['load_lineitem']
me:  1
trying: ['N1_C_O']
me:  6
trying: ['n1']
me:  6
trying: ['N2_S']
me:  6
trying: ['load_orders']
me:  3
trying: ['orders']


me:  3
trying: ['orders_filtered']
me:  6
trying: ['lineitem_filtered']
me:  6
trying: ['tpch_parent']
me:  0
trying: ['customer_filtered']
me:  6
trying: ['N2_C']
me:  6
trying: ['customer']
me:  4
trying: ['load_customer']
me:  4
trying: ['orig_output']
me:  7
trying: ['N2_C_O']
me:  6
trying: ['N1_S_L']
me:  6
trying: ['total1']
me:  6
trying: ['nation']
me:  5
trying: ['load_nation']
me:  5
trying: ['total2']
me:  6
trying: ['N1_S']
me:  6
trying: ['total']
me:  6
trying: ['N1_C']
me:  6
trying: ['load_supplier']
me:  2
trying: ['pd']
me:  0
trying: ['N2_S_L']
me:  6
trying: ['supplier']
me:  2
trying: ['STORAGE_OPTS']
me:  0
trying: ['supplier_filtered']
me:  6
trying: ['n2']
me:  6
trying: ['DATA_ROOT']
me:  0
trying: ['lineitem']


me:  1


Declaring variable tpch_parent
Declaring variable pd
Declaring variable STORAGE_OPTS
Declaring variable DATA_ROOT
Declaring variable load_lineitem
Declaring variable lineitem
Declaring variable load_supplier
Declaring variable supplier
Declaring variable load_orders
Declaring variable orders
Declaring variable customer
Declaring variable load_customer
Declaring variable nation
Declaring variable load_nation
Declaring variable N1_C_O
Declaring variable n1
Declaring variable N2_S
Declaring variable orders_filtered
Declaring variable lineitem_filtered
Declaring variable customer_filtered
Declaring variable N2_C
Declaring variable N2_C_O
Declaring variable N1_S_L
Declaring variable total1
Declaring variable total2
Declaring variable N1_S
Declaring variable total
Declaring variable N1_C
Declaring variable N2_S_L
Declaring variable supplier_filtered
Declaring variable n2
Declaring variable orig_output
